# Лабораторна робота 7

In [1]:
# %% [Р1.1] Мета, користувачі, межі системи
import sys, json, itertools
import pandas as pd, numpy as np, matplotlib, networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
matplotlib.use("Agg")
RANDOM_STATE = 42
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 80)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| networkx", nx.__version__)
SCOPE = {
    "Мета": "допомога в діагностиці апаратних і програмних несправностей настільного ПК "
            "та виборі першої ремонтної дії",
    "Користувачі": "технік служби підтримки 1-ї лінії; студент-адміністратор лабораторії",
    "Рішення": "яка ймовірна несправність; який компонент перевірити; яку дію виконати",
    "Входить": "настільний ПК: живлення, системна плата, процесор, ОЗП, накопичувачі, "
               "охолодження, відеокарта; ОС Windows: завантаження, драйвери, шкідливе ПЗ",
    "Виключено": "ноутбуки й мобільні пристрої, периферія (принтери, клавіатури), мережа, "
                 "ремонт на рівні пайки, відновлення даних, прикладне ПЗ",
}
for k, v in SCOPE.items():
    print(f"{k:12s}: {v}")

Python 3.12.16 | pandas 3.1.1 | networkx 3.7
Мета        : допомога в діагностиці апаратних і програмних несправностей настільного ПК та виборі першої ремонтної дії
Користувачі : технік служби підтримки 1-ї лінії; студент-адміністратор лабораторії
Рішення     : яка ймовірна несправність; який компонент перевірити; яку дію виконати
Входить     : настільний ПК: живлення, системна плата, процесор, ОЗП, накопичувачі, охолодження, відеокарта; ОС Windows: завантаження, драйвери, шкідливе ПЗ
Виключено   : ноутбуки й мобільні пристрої, периферія (принтери, клавіатури), мережа, ремонт на рівні пайки, відновлення даних, прикладне ПЗ


In [2]:
# %% [Р1.2] Компетентнісні питання та джерела знань
CQ = pd.DataFrame([
    ("CQ1", "Які симптоми можуть свідчити про відмову блока живлення?"),
    ("CQ2", "Які несправності належать до підсистеми пам'яті?"),
    ("CQ3", "Яка дія усуває несправність «перегрів процесора»?"),
    ("CQ4", "З яких компонентів складається системний блок (з урахуванням вкладеності)?"),
    ("CQ5", "Які несправності є програмними, а які апаратними?"),
    ("CQ6", "Скільки відомих несправностей пов'язано з кожним компонентом?"),
    ("CQ7", "Яка несправність найімовірніша при BSOD і помилках тесту пам'яті?"),
    ("CQ8", "Які випадки мають температуру процесора понад 90 °C?"),
    ("CQ9", "Які діагностичні дії потрібні для підтвердження збійних секторів накопичувача?"),
    ("CQ10", "Для яких зафіксованих випадків несправність ще не встановлено?"),
], columns=["id", "компетентнісне питання"])
print(CQ.to_string(index=False))
SOURCES = pd.DataFrame([
    ("S1", "CompTIA A+ Core 1 (220-1101) Exam Objectives, домен 5 «Hardware and Network "
           "Troubleshooting»", "стандарт сертифікації", 5, "офіційний, рецензований"),
    ("S2", "Mueller S. Upgrading and Repairing PCs, 22nd ed. Que, 2015", "підручник", 4,
     "авторитетний, частково застарілий"),
    ("S3", "Microsoft Learn: Advanced troubleshooting for Windows boot problems", "документація",
     5, "офіційна документація виробника ОС"),
    ("S4", "Документація MemTest86 (PassMark) та специфікація S.M.A.R.T. (ATA/ACS)",
     "технічна документація", 4, "вузька, але точна"),
    ("S5", "Інтерв'ю з адміністратором комп'ютерного класу (експерт)", "експерт", 3,
     "практичний досвід, суб'єктивність"),
], columns=["id", "джерело", "тип", "надійність (1–5)", "коментар"])
print(SOURCES.to_string(index=False))

  id                                                         компетентнісне питання
 CQ1                       Які симптоми можуть свідчити про відмову блока живлення?
 CQ2                               Які несправності належать до підсистеми пам'яті?
 CQ3                              Яка дія усуває несправність «перегрів процесора»?
 CQ4     З яких компонентів складається системний блок (з урахуванням вкладеності)?
 CQ5                              Які несправності є програмними, а які апаратними?
 CQ6                  Скільки відомих несправностей пов'язано з кожним компонентом?
 CQ7              Яка несправність найімовірніша при BSOD і помилках тесту пам'яті?
 CQ8                           Які випадки мають температуру процесора понад 90 °C?
 CQ9 Які діагностичні дії потрібні для підтвердження збійних секторів накопичувача?
CQ10                 Для яких зафіксованих випадків несправність ще не встановлено?
id                                                                          

In [3]:
# %% [Р2.1] Словник термінів (≥ 30): означення, синоніми, джерело
G = [  # термін, означення, синоніми, джерело
 ("персональний комп'ютер", "Обчислювальна система для індивідуального використання", "ПК; комп'ютер", "S2"),
 ("системний блок", "Корпус ПК з основними апаратними компонентами", "корпус; system unit", "S2"),
 ("системна плата", "Друкована плата, що з'єднує всі компоненти ПК", "материнська плата; motherboard", "S1"),
 ("процесор", "Мікросхема, що виконує інструкції програм", "CPU; ЦП", "S1"),
 ("оперативна пам'ять", "Енергозалежна пам'ять для даних і програм, що виконуються", "ОЗП; RAM", "S1"),
 ("накопичувач", "Пристрій постійного зберігання даних", "диск; storage", "S1"),
 ("жорсткий диск", "Накопичувач на магнітних пластинах", "HDD; вінчестер", "S2"),
 ("твердотільний накопичувач", "Накопичувач на флеш-пам'яті без рухомих частин", "SSD", "S2"),
 ("блок живлення", "Пристрій перетворення мережевої напруги у напруги живлення ПК", "БЖ; PSU", "S1"),
 ("відеокарта", "Пристрій формування зображення", "графічний адаптер; GPU", "S1"),
 ("система охолодження", "Сукупність засобів відведення тепла від компонентів", "кулер", "S2"),
 ("вентилятор", "Елемент примусової циркуляції повітря", "кулер; fan", "S2"),
 ("радіатор", "Металевий теплообмінник для розсіювання тепла", "heatsink", "S2"),
 ("термопаста", "Теплопровідний матеріал між процесором і радіатором", "термоінтерфейс", "S2"),
 ("батарейка CMOS", "Елемент живлення пам'яті налаштувань BIOS/UEFI і годинника", "CR2032", "S1"),
 ("несправність", "Стан об'єкта, за якого він не виконує хоча б одну функцію", "відмова; дефект; fault", "S1"),
 ("апаратна несправність", "Несправність фізичного компонента", "hardware fault", "S1"),
 ("програмна несправність", "Несправність ОС, драйвера чи іншого ПЗ", "software fault", "S3"),
 ("симптом", "Спостережуваний прояв несправності", "ознака; прояв", "S1"),
 ("POST", "Самотестування апаратури під час увімкнення", "power-on self-test", "S1"),
 ("звуковий сигнал POST", "Код помилки POST у вигляді послідовності сигналів", "beep code", "S2"),
 ("синій екран", "Аварійне завершення роботи Windows з кодом помилки", "BSOD; stop-помилка", "S3"),
 ("завантажувач", "Програма, що завантажує ядро ОС", "bootloader; Windows Boot Manager", "S3"),
 ("драйвер", "ПЗ, що забезпечує взаємодію ОС з пристроєм", "driver", "S3"),
 ("шкідливе програмне забезпечення", "ПЗ, створене для несанкціонованих дій", "вірус; malware", "S1"),
 ("перегрів", "Перевищення допустимої робочої температури компонента", "overheating", "S2"),
 ("збійний сектор", "Ділянка накопичувача, з якої неможливо надійно прочитати дані", "bad sector; бед-блок", "S4"),
 ("S.M.A.R.T.", "Технологія самодіагностики накопичувачів", "SMART", "S4"),
 ("тест пам'яті", "Програмна перевірка ОЗП на помилки читання/запису", "MemTest86; memtest", "S4"),
 ("артефакти зображення", "Спотворення на екрані: смуги, плями, мерехтіння", "графічні артефакти", "S1"),
 ("циклічне перезавантаження", "Повторні перезапуски ПК без завантаження ОС", "boot loop", "S3"),
 ("діагностична дія", "Дія для підтвердження чи спростування гіпотези про несправність", "перевірка; тест", "S1"),
 ("ремонтна дія", "Дія для усунення несправності", "усунення; виправлення", "S1"),
 ("профіль XMP", "Набір параметрів розгону ОЗП, що зберігається в модулі", "Extreme Memory Profile", "S5"),
 ("діагностичний випадок", "Зафіксоване звернення з набором симптомів конкретного ПК", "кейс; заявка", "S5"),
]
gloss = pd.DataFrame(G, columns=["термін", "означення", "синоніми", "джерело"])
# уніфікація: нижній регістр (крім абревіатур), однина, без дублікатів назв і синонімів
norm = gloss.термін.str.strip()
print("Термінів:", len(gloss), "| дублікатів назв:", norm.str.lower().duplicated().sum())
syn = gloss.assign(s=gloss.синоніми.str.split("; ")).explode("s")
clash = syn[syn.s.str.lower().isin(norm.str.lower())]
print("Синонім збігається з іншим терміном:", clash[["термін", "s"]].values.tolist())
POLYSEMY = {
    "кулер": "у розмовній мові — і вентилятор, і вся система охолодження; обрано: "
             "«вентилятор» (елемент), «кулер» лише синонім",
    "пам'ять": "може означати ОЗП або накопичувач; обрано: «пам'ять» = ОЗП, накопичувач — окремий термін",
    "диск": "фізичний HDD/SSD або логічний розділ; обрано: фізичний «накопичувач»",
    "несправність": "стан об'єкта (fault) або подія (failure); обрано: стан",
}
for k, v in POLYSEMY.items():
    print(f"  багатозначний «{k}»: {v}")
gloss.to_csv("glossary.csv", index=False, encoding="utf-8-sig")
print(gloss.head(12).to_string(index=False))

Термінів: 35 | дублікатів назв: 0
Синонім збігається з іншим терміном: []
  багатозначний «кулер»: у розмовній мові — і вентилятор, і вся система охолодження; обрано: «вентилятор» (елемент), «кулер» лише синонім
  багатозначний «пам'ять»: може означати ОЗП або накопичувач; обрано: «пам'ять» = ОЗП, накопичувач — окремий термін
  багатозначний «диск»: фізичний HDD/SSD або логічний розділ; обрано: фізичний «накопичувач»
  багатозначний «несправність»: стан об'єкта (fault) або подія (failure); обрано: стан
                   термін                                                     означення                       синоніми джерело
   персональний комп'ютер        Обчислювальна система для індивідуального використання                  ПК; комп'ютер      S2
           системний блок                 Корпус ПК з основними апаратними компонентами            корпус; system unit      S2
           системна плата                 Друкована плата, що з'єднує всі компоненти ПК материнська плата; moth

In [4]:
# %% [Р3.1] Таксономія is-a з критеріями поділу на кожному рівні
IS_A = {  # дочірнє поняття -> батьківське
 "Несправність": "Поняття", "Симптом": "Поняття", "Компонент": "Поняття", "Дія": "Поняття",
 # Несправність: L1 — природа; L2 — підсистема; L3 — механізм
 "Апаратна несправність": "Несправність", "Програмна несправність": "Несправність",
 "Несправність живлення": "Апаратна несправність", "Несправність пам'яті": "Апаратна несправність",
 "Несправність накопичувача": "Апаратна несправність", "Несправність охолодження": "Апаратна несправність",
 "Несправність відеопідсистеми": "Апаратна несправність",
 "Відмова блока живлення": "Несправність живлення", "Розряджена батарейка CMOS": "Несправність живлення",
 "Дефект модуля ОЗП": "Несправність пам'яті", "Поганий контакт ОЗП": "Несправність пам'яті",
 "Нестабільний розгін ОЗП": "Несправність пам'яті",
 "Збійні сектори": "Несправність накопичувача",
 "Перегрів процесора": "Несправність охолодження", "Відмова вентилятора": "Несправність охолодження",
 "Відмова відеокарти": "Несправність відеопідсистеми",
 "Несправність ОС": "Програмна несправність", "Несправність драйвера": "Програмна несправність",
 "Шкідливе ПЗ": "Програмна несправність",
 "Пошкодження завантажувача": "Несправність ОС", "Пошкодження системних файлів": "Несправність ОС",
 # Симптом: L1 — етап роботи ПК, на якому проявляється
 "Симптом увімкнення": "Симптом", "Симптом завантаження": "Симптом", "Симптом роботи": "Симптом",
 "Немає живлення": "Симптом увімкнення", "Сигнали POST": "Симптом увімкнення",
 "Немає зображення": "Симптом увімкнення",
 "Синій екран": "Симптом завантаження", "Помилка завантажувача": "Симптом завантаження",
 "Циклічне перезавантаження": "Симптом завантаження",
 "Раптове вимкнення": "Симптом роботи", "Повільна робота": "Симптом роботи",
 "Артефакти зображення": "Симптом роботи", "Скидання дати й часу": "Симптом роботи",
 # Компонент: L1 — функція; L2 — тип пристрою
 "Компонент живлення": "Компонент", "Обчислювальний компонент": "Компонент",
 "Компонент зберігання": "Компонент", "Компонент охолодження": "Компонент",
 "Комутаційний компонент": "Компонент",
 "Блок живлення": "Компонент живлення", "Батарейка CMOS": "Компонент живлення",
 "Процесор": "Обчислювальний компонент", "Відеокарта": "Обчислювальний компонент",
 "Оперативна пам'ять": "Компонент зберігання", "Накопичувач": "Компонент зберігання",
 "Жорсткий диск": "Накопичувач", "Твердотільний накопичувач": "Накопичувач",
 "Вентилятор": "Компонент охолодження", "Радіатор": "Компонент охолодження",
 "Системна плата": "Комутаційний компонент",
 # Дія: L1 — мета дії
 "Діагностична дія": "Дія", "Ремонтна дія": "Дія",
 "Тест пам'яті": "Діагностична дія", "Перевірка SMART": "Діагностична дія",
 "Вимірювання температури": "Діагностична дія", "Перевірка іншим БЖ": "Діагностична дія",
 "Заміна компонента": "Ремонтна дія", "Чищення від пилу": "Ремонтна дія",
 "Відновлення завантажувача": "Ремонтна дія", "Перевстановлення драйвера": "Ремонтна дія",
 "Антивірусна перевірка": "Ремонтна дія", "Вимкнення XMP": "Ремонтна дія",
}
CRITERIA = {"Поняття": "роль у процесі діагностики",
            "Несправність": "природа (апаратна / програмна)",
            "Апаратна несправність": "підсистема ПК", "Програмна несправність": "рівень ПЗ",
            "Несправність живлення": "механізм", "Несправність пам'яті": "механізм",
            "Несправність охолодження": "механізм", "Несправність ОС": "пошкоджений елемент",
            "Симптом": "етап роботи ПК", "Компонент": "функція компонента",
            "Компонент зберігання": "тип зберігання", "Накопичувач": "технологія",
            "Дія": "мета дії"}
T = nx.DiGraph([(p, c) for c, p in IS_A.items()])
depth = nx.shortest_path_length(T, "Поняття")
print("Понять у таксономії:", T.number_of_nodes(), "| максимальна глибина:", max(depth.values()))
for parent, crit in CRITERIA.items():
    print(f"  {parent:24s} → критерій: {crit:30s} | підкласи: {list(T.successors(parent))}")
# перевірка «X є Y» по ланцюжку
print("Приклад: «Твердотільний накопичувач» є", nx.shortest_path(T, "Поняття", "Твердотільний накопичувач")[::-1][1:])
json.dump({"is_a": IS_A, "criteria": CRITERIA}, open("taxonomy.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=1)

Понять у таксономії: 67 | максимальна глибина: 4
  Поняття                  → критерій: роль у процесі діагностики     | підкласи: ['Несправність', 'Симптом', 'Компонент', 'Дія']
  Несправність             → критерій: природа (апаратна / програмна) | підкласи: ['Апаратна несправність', 'Програмна несправність']
  Апаратна несправність    → критерій: підсистема ПК                  | підкласи: ['Несправність живлення', "Несправність пам'яті", 'Несправність накопичувача', 'Несправність охолодження', 'Несправність відеопідсистеми']
  Програмна несправність   → критерій: рівень ПЗ                      | підкласи: ['Несправність ОС', 'Несправність драйвера', 'Шкідливе ПЗ']
  Несправність живлення    → критерій: механізм                       | підкласи: ['Відмова блока живлення', 'Розряджена батарейка CMOS']
  Несправність пам'яті     → критерій: механізм                       | підкласи: ['Дефект модуля ОЗП', 'Поганий контакт ОЗП', 'Нестабільний розгін ОЗП']
  Несправність охолодження → кри

In [5]:
# %% [Р3.2] Відношення part-of (окремо від is-a)
PART_OF = [  # частина, ціле
 ("Системний блок", "Персональний комп'ютер"), ("Монітор", "Персональний комп'ютер"),
 ("Системна плата", "Системний блок"), ("Блок живлення", "Системний блок"),
 ("Накопичувач", "Системний блок"), ("Система охолодження", "Системний блок"),
 ("Відеокарта", "Системний блок"),
 ("Процесор", "Системна плата"), ("Оперативна пам'ять", "Системна плата"),
 ("Батарейка CMOS", "Системна плата"),
 ("Вентилятор", "Система охолодження"), ("Радіатор", "Система охолодження"),
 ("Термопаста", "Система охолодження"),
]
P = nx.DiGraph(PART_OF)
print("part-of ребер:", P.number_of_edges(), "| ациклічний:", nx.is_directed_acyclic_graph(P))
print("Транзитивно «Процесор» є частиною:", sorted(nx.descendants(P, "Процесор")))
json.dump(PART_OF, open("part_of.json", "w", encoding="utf-8"), ensure_ascii=False)

part-of ребер: 13 | ациклічний: True
Транзитивно «Процесор» є частиною: ["Персональний комп'ютер", 'Системна плата', 'Системний блок']


In [6]:
# %% [Р3.3] Діаграма Хассе (транзитивна редукція) для гілки «Несправність»
def hasse(G, root, path, title, w=17, h=7):
    sub = G.subgraph(nx.descendants(G, root) | {root}).copy()
    red = nx.transitive_reduction(sub)          # прибираємо транзитивні ребра
    lv = nx.shortest_path_length(red, root); rows = {}
    for n, l in lv.items(): rows.setdefault(l, []).append(n)
    pos = {}
    for l, ns in rows.items():
        ns.sort(key=lambda n: (list(nx.ancestors(red, n)) and sorted(nx.ancestors(red, n))[-1], n))
        for i, n in enumerate(ns): pos[n] = ((i + 1) / (len(ns) + 1), -l)
    fig, ax = plt.subplots(figsize=(w, h))
    nx.draw_networkx_edges(red, pos, ax=ax, arrows=False, edge_color="#666")
    for n, (x, y) in pos.items():
        ax.text(x, y, n.replace(" ", "\n", 1) if len(n) > 14 else n, ha="center", va="center",
                fontsize=8, bbox=dict(boxstyle="round", fc=["#DCEAF7", "#E2F0D9", "#FFF2CC", "#FCE4D6"][-y % 4]))
    ax.set_title(title); ax.axis("off"); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close()
    return sub.number_of_edges(), red.number_of_edges()
print("Несправність: ребер до/після редукції =", hasse(T, "Несправність", FIG / "l7_hasse_fault.png",
      "Діаграма Хассе is-a: «Несправність» (рівні: природа → підсистема → механізм)"))
print("Симптом/Компонент/Дія:", [hasse(T, r, FIG / f"l7_hasse_{i}.png", f"Діаграма Хассе is-a: «{r}»", 15, 4.5)
                                for i, r in enumerate(["Симптом", "Компонент", "Дія"])])
# part-of як окрема діаграма
fig, ax = plt.subplots(figsize=(13, 5))
pos = nx.bfs_layout(P.reverse(), "Персональний комп'ютер") if hasattr(nx, "bfs_layout") else nx.spring_layout(P, seed=42)
pos = {n: (-y, -x) for n, (x, y) in pos.items()} if hasattr(nx, "bfs_layout") else pos
nx.draw_networkx(P, pos, ax=ax, node_color="#FFF2CC", node_size=2600, font_size=7.5,
                 arrows=True, arrowstyle="-|>", edge_color="#888")
ax.set_title("Відношення part-of (стрілка: частина → ціле)"); ax.axis("off")
fig.tight_layout(); fig.savefig(FIG / "l7_partof.png", dpi=150); plt.close()

Несправність: ребер до/після редукції = (21, 21)


Симптом/Компонент/Дія: [(13, 13), (16, 16), (12, 12)]


In [7]:
# %% [Р3.4] Перевірка MECE на прикладах реальних несправностей
EXAMPLES = {  # приклад -> підходящі класи рівня «підсистема/рівень ПЗ»
 "здутий конденсатор у БЖ": ["Несправність живлення"],
 "помилка читання модуля DDR4": ["Несправність пам'яті"],
 "клацання HDD і бед-блоки": ["Несправність накопичувача"],
 "засмічений пилом радіатор": ["Несправність охолодження"],
 "смуги на екрані через відвал чипа GPU": ["Несправність відеопідсистеми"],
 "BSOD після оновлення відеодрайвера": ["Несправність драйвера"],
 "майнер у автозавантаженні": ["Шкідливе ПЗ"],
 "пошкоджено BCD після вимкнення під час оновлення": ["Несправність ОС"],
 "перегрів GPU через зупинений вентилятор відеокарти": ["Несправність охолодження", "Несправність відеопідсистеми"],
 "пошкодження файлів ОС через збійні сектори": ["Несправність накопичувача", "Несправність ОС"],
 "окислений контакт мережевого роз'єму": [],
}
mece = pd.DataFrame([(k, len(v), "; ".join(v) or "—") for k, v in EXAMPLES.items()],
                    columns=["приклад", "к-сть класів", "класи"])
mece["MECE"] = np.select([mece["к-сть класів"] == 1, mece["к-сть класів"] == 0],
                         ["OK", "прогалина (не вичерпно)"], "перетин (не взаємовиключно)")
print(mece.to_string(index=False))

                                           приклад  к-сть класів                                                  класи                        MECE
                           здутий конденсатор у БЖ             1                                  Несправність живлення                          OK
                       помилка читання модуля DDR4             1                                   Несправність пам'яті                          OK
                          клацання HDD і бед-блоки             1                              Несправність накопичувача                          OK
                         засмічений пилом радіатор             1                               Несправність охолодження                          OK
             смуги на екрані через відвал чипа GPU             1                           Несправність відеопідсистеми                          OK
                BSOD після оновлення відеодрайвера             1                                  Несправність д

In [8]:
# %% [Р4.1] База правил ЯКЩО–ТО (версія 1, як зібрано з джерел)
RULES_V1 = [  # (id, умови, висновок, джерело)
 ("R1", {"немає_живлення", "вентилятор_БЖ_не_обертається"}, "підозра_БЖ", "S1"),
 ("R2", {"підозра_БЖ", "інший_БЖ_запускає_ПК"}, "Відмова блока живлення", "S1"),
 ("R3", {"сигнали_POST_пам'ять"}, "проблема_ОЗП", "S2"),
 ("R4", {"проблема_ОЗП", "memtest_помилки"}, "Дефект модуля ОЗП", "S4"),
 ("R5", {"проблема_ОЗП", "memtest_без_помилок"}, "Поганий контакт ОЗП", "S2"),
 ("R6", {"раптове_вимкнення", "температура_CPU_висока"}, "Перегрів процесора", "S2"),
 ("R7", {"Перегрів процесора", "вентилятор_CPU_не_обертається"}, "Відмова вентилятора", "S2"),
 ("R8", {"Перегрів процесора", "пил_на_радіаторі"}, "дія: Чищення від пилу", "S5"),
 ("R9", {"синій_екран", "нещодавно_оновлено_драйвер"}, "Несправність драйвера", "S3"),
 ("R10", {"синій_екран", "memtest_помилки"}, "Дефект модуля ОЗП", "S3"),
 ("R11", {"помилка_завантажувача", "SMART_норма"}, "Пошкодження завантажувача", "S3"),
 ("R12", {"помилка_завантажувача", "SMART_переназначені_сектори"}, "Збійні сектори", "S4"),
 ("R13", {"повільна_робота", "навантаження_CPU_у_простої", "невідомі_процеси"}, "Шкідливе ПЗ", "S1"),
 ("R14", {"скидання_дати_часу"}, "Розряджена батарейка CMOS", "S1"),
 ("R15", {"Відмова блока живлення"}, "дія: Заміна компонента (БЖ)", "S1"),
 ("R16", {"Пошкодження завантажувача"}, "дія: Відновлення завантажувача", "S3"),
 ("R17", {"Несправність драйвера"}, "дія: Перевстановлення драйвера", "S3"),
 ("R18", {"Шкідливе ПЗ"}, "дія: Антивірусна перевірка", "S1"),
 # з форуму (S5, експерт): суперечить R9
 ("R19", {"синій_екран", "нещодавно_оновлено_драйвер"}, "¬Несправність драйвера", "S5"),
 # надлишкове: окремий випадок R9
 ("R20", {"синій_екран", "нещодавно_оновлено_драйвер", "вентилятор_CPU_обертається"},
  "Несправність драйвера", "S5"),
]
for rid, cond, concl, src in RULES_V1:
    print(f"{rid:4s} ЯКЩО {' І '.join(sorted(cond)):70s} ТО {concl:32s} [{src}]")

R1   ЯКЩО вентилятор_БЖ_не_обертається І немає_живлення                          ТО підозра_БЖ                       [S1]
R2   ЯКЩО підозра_БЖ І інший_БЖ_запускає_ПК                                      ТО Відмова блока живлення           [S1]
R3   ЯКЩО сигнали_POST_пам'ять                                                   ТО проблема_ОЗП                     [S2]
R4   ЯКЩО memtest_помилки І проблема_ОЗП                                         ТО Дефект модуля ОЗП                [S4]
R5   ЯКЩО memtest_без_помилок І проблема_ОЗП                                     ТО Поганий контакт ОЗП              [S2]
R6   ЯКЩО раптове_вимкнення І температура_CPU_висока                             ТО Перегрів процесора               [S2]
R7   ЯКЩО Перегрів процесора І вентилятор_CPU_не_обертається                     ТО Відмова вентилятора              [S2]
R8   ЯКЩО Перегрів процесора І пил_на_радіаторі                                  ТО дія: Чищення від пилу            [S5]
R9   ЯКЩО нещодавно_онов

In [9]:
# %% [Р4.2] Прямий ланцюг виведення з протоколом і поясненням «чому»
def forward_chain(facts, rules, verbose=True):
    facts = set(facts); why = {f: ("факт", None) for f in facts}; log = []; step = 0
    changed = True
    while changed:
        changed = False
        for rid, cond, concl, _ in rules:
            if cond <= facts and concl not in facts:
                step += 1; facts.add(concl); why[concl] = (rid, sorted(cond)); changed = True
                log.append((step, rid, " І ".join(sorted(cond)), concl))
    if verbose:
        for s, rid, c, k in log: print(f"  крок {s}: спрацювало {rid}: {c} → {k}")
    return facts, why, log

def explain(goal, why, indent=0):
    src, cond = why[goal]
    if src == "факт":
        return [" " * indent + f"• {goal} — вхідний факт"]
    out = [" " * indent + f"• {goal} — за правилом {src}, бо:"]
    for c in cond: out += explain(c, why, indent + 4)
    return out

def diagnoses(facts):
    diag = set(IS_A) & facts
    return sorted(diag), sorted(f for f in facts if f.startswith("дія:"))

In [10]:
# %% [Р4.3] Тестування на 5 сценаріях
RULES = [r for r in RULES_V1 if r[0] not in ("R19", "R20")]   # робоча версія (див. Р5)
SCENARIOS = {
 "C1 ПК не вмикається": {"немає_живлення", "вентилятор_БЖ_не_обертається", "інший_БЖ_запускає_ПК"},
 "C2 BSOD після оновлення": {"синій_екран", "нещодавно_оновлено_драйвер"},
 "C3 вимикається під навантаженням": {"раптове_вимкнення", "температура_CPU_висока", "пил_на_радіаторі"},
 "C4 не завантажується ОС": {"помилка_завантажувача", "SMART_переназначені_сектори"},
 "C5 сигнали POST, тест ОЗП чистий": {"сигнали_POST_пам'ять", "memtest_без_помилок"},
}
res = []
for name, f in SCENARIOS.items():
    print(f"\n=== {name}: факти {sorted(f)}")
    facts, why, log = forward_chain(f, RULES)
    d, a = diagnoses(facts); res.append([name, ", ".join(d) or "—", ", ".join(a) or "—", len(log)])
    for g in d: print("\n".join(explain(g, why)))
print()
print(pd.DataFrame(res, columns=["сценарій", "діагноз", "дія", "кроків"]).to_string(index=False))


=== C1 ПК не вмикається: факти ['вентилятор_БЖ_не_обертається', 'немає_живлення', 'інший_БЖ_запускає_ПК']
  крок 1: спрацювало R1: вентилятор_БЖ_не_обертається І немає_живлення → підозра_БЖ
  крок 2: спрацювало R2: підозра_БЖ І інший_БЖ_запускає_ПК → Відмова блока живлення
  крок 3: спрацювало R15: Відмова блока живлення → дія: Заміна компонента (БЖ)
• Відмова блока живлення — за правилом R2, бо:
    • підозра_БЖ — за правилом R1, бо:
        • вентилятор_БЖ_не_обертається — вхідний факт
        • немає_живлення — вхідний факт
    • інший_БЖ_запускає_ПК — вхідний факт

=== C2 BSOD після оновлення: факти ['нещодавно_оновлено_драйвер', 'синій_екран']
  крок 1: спрацювало R9: нещодавно_оновлено_драйвер І синій_екран → Несправність драйвера
  крок 2: спрацювало R17: Несправність драйвера → дія: Перевстановлення драйвера
• Несправність драйвера — за правилом R9, бо:
    • нещодавно_оновлено_драйвер — вхідний факт
    • синій_екран — вхідний факт

=== C3 вимикається під навантаженням: факти

In [11]:
# %% [Р5.1] Перевірка якості: суперечності, надлишковість, прогалини
def neg(x): return x[1:] if x.startswith("¬") else "¬" + x
def check_kb(rules):
    contra, redund = [], []
    for (a, ca, ka, _), (b, cb, kb, _) in itertools.combinations(rules, 2):
        if ca == cb and ka == neg(kb): contra.append((a, b, sorted(ca), ka, kb))
        if ka == kb and ca < cb: redund.append((b, a, "умови ширші, висновок той самий"))
        if ka == kb and cb < ca: redund.append((a, b, "умови ширші, висновок той самий"))
    return contra, redund
contra, redund = check_kb(RULES_V1)
print("Суперечності:", contra)
print("Надлишкові (правило, яке поглинається, ← загальніше):", redund)
# прогалини: кожен первинний симптом окремо та в парах із діагностичними фактами
ENTRY = ["немає_живлення", "сигнали_POST_пам'ять", "немає_зображення", "синій_екран",
         "помилка_завантажувача", "циклічне_перезавантаження", "раптове_вимкнення",
         "повільна_робота", "артефакти_зображення", "скидання_дати_часу"]
used = set().union(*[c for _, c, _, _ in RULES])
gaps = [s for s in ENTRY if s not in used]
print("Симптоми без жодного правила (прогалини):", gaps)

Суперечності: [('R9', 'R19', ['нещодавно_оновлено_драйвер', 'синій_екран'], 'Несправність драйвера', '¬Несправність драйвера')]
Надлишкові (правило, яке поглинається, ← загальніше): [('R20', 'R9', 'умови ширші, висновок той самий')]
Симптоми без жодного правила (прогалини): ['немає_зображення', 'циклічне_перезавантаження', 'артефакти_зображення']


In [12]:
# %% [Р5.2] Контрприклади та виправлена база правил (версія 2)
COUNTER = [
 ("R14", {"скидання_дати_часу", "часовий_пояс_змінено"},
  "час «скидається» через неправильний часовий пояс/синхронізацію, батарейка справна"),
 ("R6", {"раптове_вимкнення", "температура_CPU_висока", "БЖ_перевантажений"},
  "температура висока, але вимикає ПК захист перевантаженого БЖ"),
 ("R4", {"проблема_ОЗП", "memtest_помилки", "XMP_увімкнено"},
  "помилки тесту зникають після вимкнення XMP — модуль справний"),
]
for rid, f, why_txt in COUNTER:
    facts, _, _ = forward_chain(f, RULES, verbose=False)
    print(f"{rid}: факти {sorted(f)} → {diagnoses(facts)[0]}  ✗ ({why_txt})")
RULES_V2 = [r for r in RULES if r[0] not in ("R4", "R6", "R14")] + [
 ("R4'", {"проблема_ОЗП", "memtest_помилки", "XMP_вимкнено"}, "Дефект модуля ОЗП", "S4"),
 ("R4b", {"memtest_помилки", "XMP_увімкнено"}, "Нестабільний розгін ОЗП", "S5"),
 ("R4c", {"Нестабільний розгін ОЗП"}, "дія: Вимкнення XMP", "S5"),
 ("R6'", {"раптове_вимкнення", "температура_CPU_висока", "БЖ_у_нормі"}, "Перегрів процесора", "S2"),
 ("R14'", {"скидання_дати_часу", "скидаються_налаштування_BIOS"}, "Розряджена батарейка CMOS", "S1"),
 ("R21", {"немає_зображення", "вентилятори_обертаються", "інша_відеокарта_працює"}, "Відмова відеокарти", "S1"),
 ("R22", {"артефакти_зображення", "температура_GPU_висока"}, "Відмова відеокарти", "S1"),
 ("R23", {"циклічне_перезавантаження", "SMART_норма", "sfc_знайшов_помилки"}, "Пошкодження системних файлів", "S3"),
 ("R24", {"Збійні сектори"}, "дія: Заміна компонента (накопичувач)", "S4"),
 ("R25", {"Поганий контакт ОЗП"}, "дія: Чищення контактів і перевстановлення ОЗП", "S2"),
 ("R26", {"Відмова вентилятора"}, "дія: Заміна компонента (вентилятор)", "S2"),
 ("R27", {"Розряджена батарейка CMOS"}, "дія: Заміна компонента (батарейка CMOS)", "S1"),
 ("R28", {"Відмова відеокарти"}, "дія: Заміна компонента (відеокарта)", "S1"),
 ("R29", {"Дефект модуля ОЗП"}, "дія: Заміна компонента (модуль ОЗП)", "S4"),
 ("R30", {"Пошкодження системних файлів"}, "дія: Відновлення файлів (sfc /scannow, DISM)", "S3"),
]
for rid, f, _ in COUNTER:
    facts, _, _ = forward_chain(f, RULES_V2, verbose=False)
    print(f"після виправлення {rid}: {sorted(f)} → {diagnoses(facts)[0] or 'немає хибного діагнозу'}")
print("V2: правил", len(RULES_V2), "| суперечностей", len(check_kb(RULES_V2)[0]), "| надлишкових",
      len(check_kb(RULES_V2)[1]), "| прогалин",
      [s for s in ENTRY if s not in set().union(*[c for _, c, _, _ in RULES_V2])])
# діагнози, для яких немає жодної дії (прогалина 2-го типу)
diag_all = {k for _, _, k, _ in RULES_V2 if k in IS_A}
no_action = [d for d in diag_all if not any(d in c and k.startswith("дія:") for _, c, k, _ in RULES_V2)]
print("Діагнози без ремонтної дії у V2:", sorted(no_action))
# сценарії з Р4 на V2 — регресійна перевірка
reg = {n: diagnoses(forward_chain(f, RULES_V2, False)[0]) for n, f in SCENARIOS.items()}
for n, (d, a) in reg.items(): print(f"  V2 {n:34s} → {d} {a}")
# C3 тепер вимагає перевірити БЖ (уточнення R6'): додаємо факт перевірки
d, a = diagnoses(forward_chain(SCENARIOS["C3 вимикається під навантаженням"] | {"БЖ_у_нормі"}, RULES_V2, False)[0])
print("  V2 C3 + факт «БЖ_у_нормі»            →", d, a)
json.dump([(r, sorted(c), k, s) for r, c, k, s in RULES_V2], open("rules_v2.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=1)

R14: факти ['скидання_дати_часу', 'часовий_пояс_змінено'] → ['Розряджена батарейка CMOS']  ✗ (час «скидається» через неправильний часовий пояс/синхронізацію, батарейка справна)
R6: факти ['БЖ_перевантажений', 'раптове_вимкнення', 'температура_CPU_висока'] → ['Перегрів процесора']  ✗ (температура висока, але вимикає ПК захист перевантаженого БЖ)
R4: факти ['XMP_увімкнено', 'memtest_помилки', 'проблема_ОЗП'] → ['Дефект модуля ОЗП']  ✗ (помилки тесту зникають після вимкнення XMP — модуль справний)
після виправлення R14: ['скидання_дати_часу', 'часовий_пояс_змінено'] → немає хибного діагнозу
після виправлення R6: ['БЖ_перевантажений', 'раптове_вимкнення', 'температура_CPU_висока'] → немає хибного діагнозу
після виправлення R4: ['XMP_увімкнено', 'memtest_помилки', 'проблема_ОЗП'] → ['Нестабільний розгін ОЗП']
V2: правил 30 | суперечностей 0 | надлишкових 0 | прогалин []
Діагнози без ремонтної дії у V2: []
  V2 C1 ПК не вмикається                → ['Відмова блока живлення'] ['дія: Заміна ком

In [13]:
# %% [Р5.3] Узгодження експертів: κ Коена для 20 оцінок
from sklearn.metrics import cohen_kappa_score, confusion_matrix
# 20 правил/випадків оцінено двома експертами за шкалою: 2 — правило коректне,
# 1 — коректне з уточненням, 0 — некоректне. ДЕМОНСТРАЦІЙНІ оцінки — замініть реальними
# (файл kappa_ratings.csv), якщо оцінювання проводили інші експерти.
ratings = pd.read_csv("kappa_ratings.csv")
k = cohen_kappa_score(ratings.expert_1, ratings.expert_2)
kw = cohen_kappa_score(ratings.expert_1, ratings.expert_2, weights="linear")
po = (ratings.expert_1 == ratings.expert_2).mean()
print(f"n = {len(ratings)} | частка збігів p_o = {po:.2f} | κ Коена = {k:.3f} | зважена κ = {kw:.3f}")
print(pd.DataFrame(confusion_matrix(ratings.expert_1, ratings.expert_2),
                   index=["E1=0", "E1=1", "E1=2"], columns=["E2=0", "E2=1", "E2=2"]))
print("Розбіжності (обговорено у 2-му раунді Дельфі):",
      ratings.loc[ratings.expert_1 != ratings.expert_2, "item"].tolist())

n = 20 | частка збігів p_o = 0.75 | κ Коена = 0.528 | зважена κ = 0.597
      E2=0  E2=1  E2=2
E1=0     1     0     0
E1=1     1     4     2
E1=2     0     2    10
Розбіжності (обговорено у 2-му раунді Дельфі): ['R3', 'R5', 'R10', 'R14', 'R18']
